# Unified Installment Extract Test (ApplicationDate)

Target output columns:
`Year`, `New/Return`, `W/S/M`, `Install #`, `Loan Cnt`, `Default Cnt`, `Paid-off Cnt`.

In [1]:
import pandas as pd
from pathlib import Path

In [21]:
# Load source dataset (ApplicationDate-based)
data = pd.read_csv('/Users/starsrain/feb2026_concord/feb2026_harvey_files/defaulted_install_raw/2023_2025_raw_v4.csv')
print('Loaded shape:', data.shape)
data.head()

Loaded shape: (3937710, 23)


,LoanID,OriginatedAmount,AppYear,AppMonth,AppWeek,TotalRealizedPayin,InstallmentNumber,PaidOffPaymentAmount,PaymentType,Payment_Number,...,PaymentID,Application_ID,PortFolioID,LoanStatus,NewlyScored,Accepted,Originated,OriginationDate,numOfReturn,numOfPayment
0,I1529465-0,800.0,2023,1,1,190.0,7,272.0,Installment Pmt,16,...,69756122,104209686,5,C,0,1,1,2023-01-03 07:47:02.520,1.0,2.0
1,I1529465-0,800.0,2023,1,1,190.0,8,248.0,Installment Pmt,17,...,69499464,104209686,5,C,0,1,1,2023-01-03 07:47:02.520,1.0,2.0
2,I1529465-0,800.0,2023,1,1,190.0,7,272.0,Installment Pmt,18,...,69756166,104209686,5,C,0,1,1,2023-01-03 07:47:02.520,1.0,2.0
3,I1529465-0,800.0,2023,1,1,190.0,8,248.0,Installment Pmt,19,...,69756123,104209686,5,C,0,1,1,2023-01-03 07:47:02.520,1.0,2.0
4,I1529465-0,800.0,2023,1,1,190.0,9,194.0,Installment Pmt,20,...,69499465,104209686,5,C,0,1,1,2023-01-03 07:47:02.520,1.0,2.0


In [4]:
data.columns

Index(['LoanID', 'OriginatedAmount', 'AppYear', 'AppMonth', 'AppWeek',
       'TotalRealizedPayin', 'InstallmentNumber', 'PaidOffPaymentAmount',
       'PaymentType', 'Payment_Number', 'PaymentStatus', 'CustType',
       'Frequency', 'PaymentID', 'Application_ID', 'PortFolioID', 'LoanStatus',
       'NewlyScored', 'Accepted', 'Originated', 'OriginationDate',
       'numOfReturn', 'numOfPayment'],
      dtype='object')

In [14]:
data['PaymentStatus'].value_counts()

PaymentStatus
V    1496449
D     887629
T     761486
R     661792
A      97730
P      31688
L        925
2          6
-          4
           1
Name: count, dtype: int64

### Final Unified Cohort Summary (v3 - status-filtered first)

This version filters `PaymentStatus` to `('D','R','A','T','P','~')` before building all metrics and uses installment-level terminal status logic.

In [ ]:
# Final unified cohort summary table (v3)
work = data.copy()

# Normalize required fields
work['Year'] = pd.to_numeric(work['AppYear'], errors='coerce')
work['InstallmentNumber'] = pd.to_numeric(work['InstallmentNumber'], errors='coerce')
work['Payment_Number_num'] = pd.to_numeric(work['Payment_Number'], errors='coerce')
work['New/Return'] = work['CustType'].astype(str).str.strip().str.upper()
work['W/S/M'] = work['Frequency'].astype(str).str.strip()
work['PaymentStatus'] = work['PaymentStatus'].astype(str).str.strip()

# Filter statuses first, as requested
valid_statuses = {'D', 'R', 'A', 'T', 'P', '~'}
work = work[work['PaymentStatus'].isin(valid_statuses)].copy()

# Keep only rows valid for this summary
base = work.dropna(subset=['LoanID', 'Year', 'InstallmentNumber', 'Payment_Number_num', 'New/Return', 'W/S/M']).copy()
base['Year'] = base['Year'].astype(int)
base['InstallmentNumber'] = base['InstallmentNumber'].astype(int)
base = base[base['New/Return'].isin(['NEW', 'RETURN'])]

key_cols = ['Year', 'New/Return', 'W/S/M', 'InstallmentNumber']
default_statuses = {'R', 'P', 'A', 'T', '~'}

# Loan Cnt: distinct loans per cohort key
loan_cnt = (
    base.groupby(key_cols, as_index=False)
    .agg(**{'Loan Cnt': ('LoanID', 'nunique')})
)

# Build installment-level terminal candidates using max Payment_Number only
installment_terminal_candidates = (
    base.groupby(['LoanID', 'InstallmentNumber'], as_index=False)['Payment_Number_num']
    .max()
    .rename(columns={'Payment_Number_num': 'MaxPaymentNumber'})
)

installment_terminal_rows = base.merge(
    installment_terminal_candidates,
    on=['LoanID', 'InstallmentNumber'],
    how='inner'
)
installment_terminal_rows = installment_terminal_rows[
    installment_terminal_rows['Payment_Number_num'] == installment_terminal_rows['MaxPaymentNumber']
].copy()

# If ties exist at max Payment_Number, resolve to one row per loan+installment.
# Rule selected: default statuses win over D.
installment_terminal_rows['StatusRank'] = 0
installment_terminal_rows.loc[
    installment_terminal_rows['PaymentStatus'] == 'D',
    'StatusRank'
] = 1
installment_terminal_rows.loc[
    installment_terminal_rows['PaymentStatus'].isin(default_statuses),
    'StatusRank'
] = 2

installment_terminal_rows = (
    installment_terminal_rows
    .sort_values(['LoanID', 'InstallmentNumber', 'StatusRank'])
    .groupby(['LoanID', 'InstallmentNumber'], as_index=False)
    .tail(1)
    .copy()
)

# Last_Install: per-loan highest installment from installment-level terminal rows
loan_last_rows = (
    installment_terminal_rows
    .sort_values(['LoanID', 'InstallmentNumber', 'Payment_Number_num'])
    .groupby('LoanID', as_index=False)
    .tail(1)
    .copy()
)
last_install_cnt = (
    loan_last_rows.groupby(key_cols, as_index=False)
    .agg(Last_Install=('LoanID', 'nunique'))
)

default_cnt = (
    installment_terminal_rows[installment_terminal_rows['PaymentStatus'].isin(default_statuses)]
    .groupby(key_cols, as_index=False)
    .agg(**{'Default Cnt': ('LoanID', 'nunique')})
)

paid_cnt = (
    installment_terminal_rows[installment_terminal_rows['PaymentStatus'] == 'D']
    .groupby(key_cols, as_index=False)
    .agg(**{'Paid_Cnt': ('LoanID', 'nunique')})
)

# Merge all metrics
summary = (
    loan_cnt
    .merge(last_install_cnt, on=key_cols, how='left')
    .merge(default_cnt, on=key_cols, how='left')
    .merge(paid_cnt, on=key_cols, how='left')
)

for col in ['Last_Install', 'Default Cnt', 'Paid_Cnt']:
    summary[col] = summary[col].fillna(0).astype(int)

summary['Default Ratio'] = summary['Default Cnt'] / summary['Loan Cnt']
summary['Paid Ratio'] = summary['Paid_Cnt'] / summary['Loan Cnt']

extract_v3 = summary.sort_values(key_cols).reset_index(drop=True)
extract_v3.head()

,Year,New/Return,W/S/M,InstallmentNumber,Loan Cnt,Last_Install,Default Cnt,Paid_Cnt,Default Ratio,Paid Ratio
0,2023,NEW,B,1,19728,1523,3475,16253,0.176146,0.823854
1,2023,NEW,B,2,18235,706,5423,12812,0.297395,0.702605
2,2023,NEW,B,3,17528,461,6921,10607,0.394854,0.605146
3,2023,NEW,B,4,17066,391,7964,9102,0.466659,0.533341
4,2023,NEW,B,5,16672,387,9307,7365,0.558241,0.441759


In [25]:
# Spot-check: installment-level terminal logic on a sample loan
sample_loan_id = 'I2512207-0'

sample_rows = base[base['LoanID'] == sample_loan_id].copy()
sample_terminal = installment_terminal_rows[installment_terminal_rows['LoanID'] == sample_loan_id].copy()
sample_last_install = loan_last_rows[loan_last_rows['LoanID'] == sample_loan_id].copy()

print('Sample raw rows after status pre-filter:', len(sample_rows))
print('Sample statuses present:', sorted(sample_rows['PaymentStatus'].dropna().unique().tolist()))
print('Sample installment-level terminal rows:', len(sample_terminal))

sample_rows.sort_values(['InstallmentNumber', 'Payment_Number_num'])[
    ['LoanID', 'InstallmentNumber', 'Payment_Number_num', 'PaymentStatus', 'Year', 'New/Return', 'W/S/M']
].head(30)

print('\nResolved terminal rows (one per LoanID + InstallmentNumber):')
sample_terminal.sort_values(['InstallmentNumber'])[
    ['LoanID', 'InstallmentNumber', 'Payment_Number_num', 'PaymentStatus', 'StatusRank', 'Year', 'New/Return', 'W/S/M']
]

print('\nLoan-level last installment row derived from installment terminal rows:')
sample_last_install[
    ['LoanID', 'InstallmentNumber', 'Payment_Number_num', 'PaymentStatus', 'Year', 'New/Return', 'W/S/M']
]

Sample raw rows after status pre-filter: 0
Sample statuses present: []
Sample installment-level terminal rows: 0

Resolved terminal rows (one per LoanID + InstallmentNumber):

Loan-level last installment row derived from installment terminal rows:


,LoanID,InstallmentNumber,Payment_Number_num,PaymentStatus,Year,New/Return,W/S/M


In [ ]:
# Audit Paid_Cnt composition for a cohort (why Paid_Cnt can be large)
audit_year = 2023
audit_cust = 'NEW'
audit_freq = 'B'
audit_installments = [1, 2, 3, 4, 5]

loan_last_install = (
    installment_terminal_rows.groupby('LoanID', as_index=False)['InstallmentNumber']
    .max()
    .rename(columns={'InstallmentNumber': 'LoanLastInstallment'})
)

paid_rows = installment_terminal_rows[installment_terminal_rows['PaymentStatus'] == 'D'].copy()
paid_rows = paid_rows.merge(loan_last_install, on='LoanID', how='left')
paid_rows['IsLoanLastInstallment'] = paid_rows['InstallmentNumber'] == paid_rows['LoanLastInstallment']

cohort_paid = paid_rows[
    (paid_rows['Year'] == audit_year) &
    (paid_rows['New/Return'] == audit_cust) &
    (paid_rows['W/S/M'] == audit_freq) &
    (paid_rows['InstallmentNumber'].isin(audit_installments))
].copy()

decomp = (
    cohort_paid.groupby('InstallmentNumber', as_index=False)
    .agg(
        Paid_Cnt=('LoanID', 'nunique'),
        Paid_And_LastInstall=('IsLoanLastInstallment', 'sum')
    )
    .sort_values('InstallmentNumber')
)

decomp['Paid_But_Not_LastInstall'] = decomp['Paid_Cnt'] - decomp['Paid_And_LastInstall']
print('Paid_Cnt decomposition for cohort:', audit_year, audit_cust, audit_freq)
print(decomp)

print('\nSample loans counted in Paid_Cnt at installment 1:')
print(
    cohort_paid[cohort_paid['InstallmentNumber'] == 1][
        ['LoanID', 'InstallmentNumber', 'Payment_Number_num', 'PaymentStatus', 'LoanLastInstallment', 'IsLoanLastInstallment']
    ]
    .sort_values(['LoanLastInstallment', 'LoanID'])
    .head(20)
)

print('\nSample loans counted in Paid_Cnt at installment 2:')
print(
    cohort_paid[cohort_paid['InstallmentNumber'] == 2][
        ['LoanID', 'InstallmentNumber', 'Payment_Number_num', 'PaymentStatus', 'LoanLastInstallment', 'IsLoanLastInstallment']
    ]
    .sort_values(['LoanLastInstallment', 'LoanID'])
    .head(20)
)